In [1]:
#データの読み込み
#boolで削除
#プロテインはやらない
#errorの削除
#denseの表示
#formatの作成

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
file_path = "/data/yamane/GPCRclassAData/"
fle_name = ["chemblSmiles.csv","classA_ligand_binary_202110.csv","id2seq.csv"]

In [4]:
smiles_df = pd.read_csv(file_path+fle_name[0]).drop("Unnamed: 0",axis = 1)
origin_df = pd.read_csv(file_path+fle_name[1]).drop("Unnamed: 0",axis = 1)
seq_df = pd.read_csv(file_path+fle_name[2]).drop("Unnamed: 0",axis = 1)

print("interaction : ",len(origin_df))
print("protein_num : ",len(set(origin_df["UniProt ID"].to_list())))
print("ligand_num : ",len(set(origin_df["Database Ligand ID"].to_list())))

interaction :  339147
protein_num :  504
ligand_num :  144721


In [5]:
origin_df["Interaction_type"].value_counts()

0    234602
1    104545
Name: Interaction_type, dtype: int64

In [6]:
print("interaction : ",len(origin_df))
print("protein_num : ",len(set(origin_df["UniProt ID"].to_list())))
print("ligand_num : ",len(set(origin_df["Database Ligand ID"].to_list())))

interaction :  339147
protein_num :  504
ligand_num :  144721


In [7]:
error_smiles = smiles_df[smiles_df["smiles"] == "error"]["CHEMBL_ID"].to_list()

In [8]:
ex = []
for i in range(len(origin_df)):
  cid = origin_df.iloc[i]["Database Ligand ID"]
  if cid in error_smiles:
    ex.append(1)
  else:
    ex.append(0)

In [9]:
#smilesのエラーが出たものを削除
origin_df["ex"] = ex
origin_df = origin_df[origin_df["ex"] == 0].reset_index().drop("index",axis = 1)
origin_df = origin_df.drop("ex",axis = 1)

In [11]:
#denseの計算
origin_df.head()
print("interaction : ",len(origin_df))
print("protein_num : ",len(set(origin_df["UniProt ID"].to_list())))
print("ligand_num : ",len(set(origin_df["Database Ligand ID"].to_list())))
origin_df.to_csv("origin.csv")

interaction :  337115
protein_num :  504
ligand_num :  143807


In [13]:
# dense_dic = {}
# for name ,it in zip(origin_df["Database Ligand ID"].to_list(),origin_df["Interaction_type"].to_list()):
#   if name not in dense_dic:
#     dense_dic[name] = [it]
#   else:
#     dense_dic[name].append(it)

# import math

# dense = []
# for name in dense_dic:
#   data = dense_dic[name]
#   count1 = data.count(1)
#   count0 = data.count(0)
#   dense.append(math.log10(count1/count0))

# from scipy.stats import kde

# plt.figure(figsize=(12, 9), dpi=50)
# data = dense
# density = kde.gaussian_kde(data)
# x = np.linspace(-2,2,300)
# y=density(x)

# plt.plot(x, y)
# plt.title("Density Plot of the data")
# #plt.savefig("/home/yamane/transformerCPI/img/density.png")
# plt.show()



In [35]:

# tcpi_format = []
# for i in range(len(origin_df)):
#   cid = origin_df.iloc[i]["Database Ligand ID"]
#   smiles = smiles_df[smiles_df["CHEMBL_ID"] == cid]["smiles"].values[0]
#   pid = origin_df.iloc[i]["UniProt ID"]
#   seq = seq_df[seq_df["UniProt ID"] == pid]["sequence"].values[0]
#   it = origin_df.iloc[i]["Interaction_type"]
#   tcpi_format.append(" ".join([smiles,seq,str(it)]))



In [36]:
with open("format.dump","rb") as f:
  tcpi_format = pickle.load(f)

In [37]:
def shuffle_dataset(dataset, seed):
    np.random.seed(seed)
    np.random.shuffle(dataset)
    return dataset


def split_dataset(dataset, ratio):
    n = int(ratio * len(dataset))
    dataset_1, dataset_2 = dataset[:n], dataset[n:]
    return dataset_1, dataset_2


In [38]:
dataset = shuffle_dataset(tcpi_format, 0)
train,test = split_dataset(dataset, 0.8)


In [39]:
origin_df.head()

,UniProt ID,InChI Key,Parameter,Value,Unit,Database Source,Database Target ID,Database Ligand ID,Reference,Interaction_type
0,P26684,SBACFWNGSKLATN-UHFFFAOYSA-N,IC50,2100,nM,ChEMBL,CHEMBL4566,CHEMBL352241,10098676,0
1,P32745,SEKPCRFTCJTKMS-CZNROWNISA-N,IC50,>1000,nM,ChEMBL,CHEMBL2028,CHEMBL1161332,14667212,0
2,P56481,UVCYUMJOUAVORG-NQDKFOQASA-N,IC50,12.2,nM,ChEMBL,CHEMBL2854,CHEMBL342932,9438020,1
3,P11229,HSFWTAVXXXJJRW-CAOOACKPSA-N,IC50,401,nM,ChEMBL,CHEMBL216,CHEMBL3600972,26077492,1
4,P41145,FGXWKSZFVQUSTL-UHFFFAOYSA-N,IC50,6994,nM,ChEMBL,CHEMBL237,CHEMBL219916,DrugMatrix in vitro pharmacology data,0


In [40]:
print("interaction : ",len(origin_df))
print("protein_num : ",len(set(origin_df["UniProt ID"].to_list())))
print("ligand_num : ",len(set(origin_df["Database Ligand ID"].to_list())))
print("train_num : test_num = ",len(train),len(test))

interaction :  106354
protein_num :  449
ligand_num :  23242
train_num : test_num =  85083 21271


In [41]:
with open("/home/yamane/transformerCPI/data/shuffle_classA/shuffle_train.txt","r") as f:
  data_list = f.read().strip().split('\n')
data_list = [d for d in data_list if '.' not in d.strip().split()[0]]
N2 = len(data_list)

In [42]:
train_count = []
for i in range(len(data_list)):
  train_count.append(int(data_list[i].split(" ")[-1]))

In [43]:
with open("/home/yamane/transformerCPI/data/shuffle_classA/shuffle_test.txt","r") as f:
  data_list = f.read().strip().split('\n')
data_list = [d for d in data_list if '.' not in d.strip().split()[0]]
N1 = len(data_list)

In [44]:
test_count = []
for i in range(len(data_list)):
  test_count.append(int(data_list[i].split(" ")[-1]))

In [45]:
print(train_count.count(0),train_count.count(1),N2)
print(test_count.count(0),test_count.count(1),N1)

48329 33001 81330
12014 8264 20278


In [46]:
len(origin_df)

106354

In [47]:
N1+N2

101608